# Sponsor OS — train the lead ranker (Smart Score v2)

Trains a **calibrated XGBoost classifier** on logged outcomes and writes
`leads.ml_score` + a `ranker_runs` audit row. The Lead Board shows the result
as "Smart Score (beta)" **alongside** the transparent Evidence Score — never
instead of it.

**Hard gates (do not remove):**
1. Refuses below **50 real outcomes** (non-demo, non-voided) — the brief's rule.
2. Refuses to upload if 5-fold **CV AUC ≤ 0.55** — never ship a coin flip.

**Why a classifier, not learning-to-rank:** at 50–200 samples an LTR objective
is statistical theater. We rank by calibrated P(positive). Revisit a pairwise
LTR objective when real outcomes exceed ~500.

## ⚠️ Selection bias — read before trusting the number
This model trains only on leads we **chose to contact** — a population already
filtered by Evidence Score and human judgment. It therefore learns
*P(positive | we pursued)*, *conversion-if-pursued*, **not** absolute brand
affinity. And once Smart Score influences who gets contacted, it curates its
own future training data. At chapter scale there is no exploration budget to
fix this properly; the working mitigation is that the Evidence Score never
disappears from the UI. This paragraph is repeated in the `ranker_runs` model
card on purpose — future committees: do not over-trust this score.

In [ ]:
%pip install -q xgboost scikit-learn pandas supabase python-dotenv

In [ ]:
RUN_SYNTHETIC_DEMO = False  # True = dry-run the pipeline on fake data, NO upload
MIN_OUTCOMES = 50
MIN_AUC = 0.55
OVERRIDE = ""  # type I_UNDERSTAND to bypass gates (don't)

import datetime
MODEL_VERSION = "ranker-v2-" + datetime.date.today().strftime("%Y%m%d")
POSITIVE = {"replied", "meeting", "signed"}
NEGATIVE = {"ghosted", "rejected"}  # sent-only = still in flight, excluded

In [ ]:
import os
import numpy as np
import pandas as pd

if RUN_SYNTHETIC_DEMO:
    rng = np.random.default_rng(7)
    n = 120
    frame = pd.DataFrame({
        "lead_id": np.arange(n),
        "evidence_score": rng.uniform(10, 95, n),
        "evidence_count": rng.integers(1, 6, n),
        "region_share": rng.uniform(0, 1, n),
        "days_since_evidence": rng.uniform(0, 400, n),
    })
    logit = 0.04 * frame.evidence_score + 1.2 * frame.region_share - 0.004 * frame.days_since_evidence - 2.2
    frame["label"] = (rng.uniform(size=n) < 1 / (1 + np.exp(-logit))).astype(int)
    print(f"SYNTHETIC dry run: {n} fake leads, {frame.label.sum()} positives. No upload possible.")
else:
    from getpass import getpass
    try:
        from dotenv import load_dotenv; load_dotenv()
    except Exception:
        pass
    from supabase import create_client
    url = os.environ.get("SUPABASE_URL") or getpass("SUPABASE_URL: ")
    key = os.environ.get("SUPABASE_SERVICE_KEY") or getpass("SUPABASE_SERVICE_KEY: ")
    client = create_client(url, key)

    outcomes = pd.DataFrame(client.table("outcomes").select(
        "lead_id, event, voided, leads(is_demo)").execute().data)
    outcomes = outcomes[~outcomes.voided]
    outcomes["is_demo"] = outcomes.leads.apply(lambda l: bool((l or {}).get("is_demo")))
    outcomes = outcomes[~outcomes.is_demo]  # purity: practice taps never train

    labels = {}
    for lead_id, group in outcomes.groupby("lead_id"):
        events = set(group.event)
        if events & POSITIVE:
            labels[lead_id] = 1
        elif events & NEGATIVE:
            labels[lead_id] = 0
        # sent-only: excluded — still in flight

    N_REAL = len(outcomes)
    print(f"{N_REAL} real outcomes, {len(labels)} labeled leads "
          f"({sum(labels.values())} positive)")
    if N_REAL < MIN_OUTCOMES and OVERRIDE != "I_UNDERSTAND":
        raise SystemExit(
            f"GATE: only {N_REAL} real outcomes; the ranker activates at {MIN_OUTCOMES}. "
            "Keep logging — the Evidence Score is doing its job until then.")

    leads = pd.DataFrame(client.table("leads").select(
        "id, evidence_score, brand_id, is_demo").eq("is_demo", False).execute().data)
    evidence = pd.DataFrame(client.table("evidence").select(
        "brand_id, region_match, detected_at").execute().data)
    evidence["detected_at"] = pd.to_datetime(evidence.detected_at, utc=True)
    now = pd.Timestamp.now(tz="UTC")
    agg = evidence.groupby("brand_id").agg(
        evidence_count=("region_match", "size"),
        region_share=("region_match", "mean"),
        days_since_evidence=("detected_at", lambda s: (now - s.max()).days),
    ).reset_index()
    frame = leads.merge(agg, on="brand_id", how="left").fillna(
        {"evidence_count": 0, "region_share": 0, "days_since_evidence": 999})
    frame["label"] = frame.id.map(labels)
    frame = frame.rename(columns={"id": "lead_id"})
    print(f"feature matrix: {len(frame)} leads, {frame.label.notna().sum()} labeled")

In [ ]:
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier

FEATURES = ["evidence_score", "evidence_count", "region_share", "days_since_evidence"]
train = frame[frame.label.notna()]
X, y = train[FEATURES], train.label.astype(int)

model = XGBClassifier(n_estimators=80, max_depth=3, learning_rate=0.1,
                      subsample=0.9, eval_metric="logloss")
CV_AUC = float(np.mean(cross_val_score(model, X, y, cv=5, scoring="roc_auc")))
model.fit(X, y)
IMPORTANCE = dict(zip(FEATURES, [round(float(v), 4) for v in model.feature_importances_]))
print(f"5-fold CV AUC = {CV_AUC:.3f}  (gate: > {MIN_AUC})")
print("feature importance:", IMPORTANCE)

In [ ]:
if RUN_SYNTHETIC_DEMO:
    raise SystemExit("Synthetic dry run complete — pipeline works, nothing uploaded.")
if CV_AUC <= MIN_AUC and OVERRIDE != "I_UNDERSTAND":
    raise SystemExit(
        f"GATE: CV AUC {CV_AUC:.3f} <= {MIN_AUC}. This model cannot beat a coin "
        "flip with margin — refusing to ship scores. More/cleaner outcomes needed.")

MODEL_CARD = (
    f"{MODEL_VERSION}: XGBoost classifier ranked by calibrated P(positive). "
    f"Trained on {N_REAL} real outcomes ({len(train)} labeled leads), 5-fold CV "
    f"AUC {CV_AUC:.3f}. SELECTION BIAS: trained on contacted leads only — scores "
    "reflect conversion-if-pursued, NOT absolute affinity, and once this score "
    "steers outreach it curates its own training data. Keep using Evidence Score "
    "alongside. Sent-only leads excluded as in-flight."
)

scores = model.predict_proba(frame[FEATURES])[:, 1] * 100
for lead_id, score in zip(frame.lead_id, scores):
    client.table("leads").update({"ml_score": round(float(score), 1)}).eq("id", int(lead_id)).execute()
client.table("ranker_runs").upsert({
    "model_version": MODEL_VERSION, "n_outcomes": int(N_REAL),
    "cv_auc": round(CV_AUC, 4), "feature_importance": IMPORTANCE,
    "model_card": MODEL_CARD, "status": "active",
}, on_conflict="model_version").execute()
print(f"Wrote ml_score for {len(frame)} leads + ranker_runs {MODEL_VERSION}.")
print(MODEL_CARD)